In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.linear_model import LinearRegression, Ridge # Adăugat Ridge aici
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

training_data = np.load('data/training_data.npy')
prices = np.load('data/prices.npy')
training_data, prices = shuffle(training_data, prices, random_state=0)

def normalize_data(train_data, test_data):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_data)
    test_scaled = scaler.transform(test_data)
    return train_scaled, test_scaled


linear_regression_model = LinearRegression()
kf = KFold(n_splits=3)

def linear_regression_ex():
    mse_scores = []
    mae_scores = []

    for train_index, val_index in kf.split(training_data):
        X_train, X_val = training_data[train_index], training_data[val_index]
        y_train, y_val = prices[train_index], prices[val_index]

        X_train_scaled, X_val_scaled = normalize_data(X_train, X_val)

        linear_regression_model.fit(X_train_scaled, y_train)
        predictions = linear_regression_model.predict(X_val_scaled)

        mse_scores.append(mean_squared_error(y_val, predictions))
        mae_scores.append(mean_absolute_error(y_val, predictions))

    mean_mse = np.mean(mse_scores)
    mean_mae = np.mean(mae_scores)

    print(f"Valoarea medie MSE după 3-fold CV (Linear): {mean_mse:.4f}")
    print(f"Valoarea medie MAE după 3-fold CV (Linear): {mean_mae:.4f}")

def ridge_ex():
    best_a = None
    best_mean_mse = float('inf')

    for a in [1, 10, 100, 1000]:
        ridge = Ridge(alpha=a)

        mse_scores = []
        mae_scores = []

        for train_index, val_index in kf.split(training_data):
            X_train, X_val = training_data[train_index], training_data[val_index]
            y_train, y_val = prices[train_index], prices[val_index]

            X_train_scaled, X_val_scaled = normalize_data(X_train, X_val)
            ridge.fit(X_train_scaled, y_train)
            predictions = ridge.predict(X_val_scaled)

            mse_scores.append(mean_squared_error(y_val, predictions))
            mae_scores.append(mean_absolute_error(y_val, predictions))

        mean_mse = np.mean(mse_scores)
        mean_mae = np.mean(mae_scores)

        print(f"Alpha = {a} -> MSE mediu: {mean_mse:.4f}, MAE mediu: {mean_mae:.4f}")

        if mean_mse < best_mean_mse:
            best_mean_mse = mean_mse
            best_a = a

    print(f"\nCel mai bun alpha selectat: {best_a} cu MSE mediu: {best_mean_mse:.4f}")
    return best_a

def ridge_regression_final(alpha):
    scaler = StandardScaler()
    training_data_scaled = scaler.fit_transform(training_data)

    final_ridge = Ridge(alpha=alpha)
    final_ridge.fit(training_data_scaled, prices)

    coeficienți = final_ridge.coef_
    bias = final_ridge.intercept_

    print(f"\n--- Rezultate finale pentru Alpha = {alpha} ---")
    print(f"Bias: {bias:.4f}")

    atribute_sortate = np.argsort(np.abs(coeficienți))[::-1]

    print(f"Cel mai semnificativ atribut: Atributul {atribute_sortate[0]} (coef: {coeficienți[atribute_sortate[0]]:.4f})")
    print(f"Al doilea cel mai semnificativ: Atributul {atribute_sortate[1]} (coef: {coeficienți[atribute_sortate[1]]:.4f})")
    print(f"Cel mai puțin semnificativ: Atributul {atribute_sortate[-1]} (coef: {coeficienți[atribute_sortate[-1]]:.4f})")

# Rulăm exercițiile
linear_regression_ex()
print("-" * 50)
best_a = ridge_ex()
ridge_regression_final(best_a)

Valoarea medie MSE după 3-fold CV (Linear): 3.1674
Valoarea medie MAE după 3-fold CV (Linear): 1.3196
--------------------------------------------------
Alpha = 1 -> MSE mediu: 3.1674, MAE mediu: 1.3196
Alpha = 10 -> MSE mediu: 3.1673, MAE mediu: 1.3194
Alpha = 100 -> MSE mediu: 3.1723, MAE mediu: 1.3186
Alpha = 1000 -> MSE mediu: 3.4332, MAE mediu: 1.3666

Cel mai bun alpha selectat: 10 cu MSE mediu: 3.1673

--- Rezultate finale pentru Alpha = 10 ---
Bias (Intercept): 5.6951
Cel mai semnificativ atribut: Atributul 0 (coef: 1.6635)
Al doilea cel mai semnificativ: Atributul 4 (coef: 1.3357)
Cel mai puțin semnificativ: Atributul 7 (coef: 0.0000)
